In [ ]:
import sys
import os

# Визначаємо шлях до каталогу project (двома рівнями вище)
module_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'code'))
if module_path not in sys.path:
    sys.path.append(module_path)

import data_exploration as exploration
import data_cleaning as cleaning
import data_construction as construction

import pandas as pd

# Data Understanding

In [ ]:
multiple_apps = [
    'Flutter_UI-Metrics_TimeSeries_MultipleApps_1.csv',
    'Flutter_UI-Metrics_TimeSeries_MultipleApps_2.csv',
    'Flutter_UI-Metrics_TimeSeries_MultipleApps_3.csv',
    'Native_UI-Metrics_TimeSeries_MultipleApps_1.csv',
    'Native_UI-Metrics_TimeSeries_MultipleApps_2.csv',
    'Native_UI-Metrics_TimeSeries_MultipleApps_3.csv'
]

usage_delays = [
    'Flutter_UI-Metrics_TimeSeries_UsageDelays_1.csv',
    'Flutter_UI-Metrics_TimeSeries_UsageDelays_2.csv',
    'Flutter_UI-Metrics_TimeSeries_UsageDelays_3.csv',
    'Native_UI-Metrics_TimeSeries_UsageDelays_1.csv',
    'Native_UI-Metrics_TimeSeries_UsageDelays_2.csv',
    'Native_UI-Metrics_TimeSeries_UsageDelays_3.csv'
]

csv_directory = 'data/external/EXP-2021'

selected_csv = multiple_apps[1]
#selected_csv = usage_delays[0]

csv_abs_path = os.path.abspath(os.path.join(os.getcwd(), '../..', csv_directory, selected_csv))

data = pd.read_csv(csv_abs_path)

data.drop(
    columns=[
        'time_end_sec',
        'percentile_50_ms',
        'percentile_90_ms',
        'percentile_95_ms',
        'percentile_99_ms'
    ],
    inplace=True
)

data.head()

In [ ]:
data.describe()

In [ ]:
data.info()

In [ ]:
exploration.show_time_series_charts(data)

In [ ]:
exploration.show_distributions(data)

In [ ]:
exploration.show_heatmap(data)

In [ ]:
exploration.show_boxplots(data)

# Data Preparation

In [ ]:
data.drop(
    columns=[
        'launch_time', # Видаляємо, оскільки відсутній в більше, ніж 50% випадків
        'draw_time_median', # Видаляємо, оскільки вважаємо її непотрібною
        'total_frames', # Видаляємо, оскільки не має відношення до старіння
        'total_frames_median', # Видаляємо, оскільки не має відношення до старіння
        'janky_ratio', # Видаляємо, оскільки тут можлива помилка в обчисленнях колонки
        'janky_ratio_median', # Видаляємо, оскільки тут можлива помилка в обчисленнях колонки
        'janky_frames_median', # Видаляємо, оскільки вважаємо її непотрібною
        'high_input_latency_count' # Видаляємо, оскільки відсутня кореляція з іншими метриками
    ], 
    inplace=True
)

In [ ]:
data = cleaning.apply_z_score(
    data, 
    [
        'draw_time_avg', 
        'janky_frames'
    ]
)

In [ ]:
data = construction.add_moving_averages(
    data, 
    columns=[
        'draw_time_avg', 
        'janky_frames'
    ]
)

In [ ]:
exploration.show_time_series_charts(data)

In [ ]:
exploration.show_boxplots(data)

In [ ]:
exploration.show_heatmap(data)

In [ ]:
csv_output_directory = 'data/processed/EXP-2021'

csv_output_abs_path = os.path.abspath(os.path.join(os.getcwd(), '..', csv_output_directory, selected_csv))

data.to_csv(csv_output_abs_path, index=False)